# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [1]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"
model = SentenceTransformer(MODEL_NAME, device="cuda")

C:\Users\FREDDY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2522.21it/s]


In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [3]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [4]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [5]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [6]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3537.95it/s]


In [7]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

Batches: 100%|██████████| 4944/4944 [08:47<00:00,  9.38it/s]


In [8]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [9]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [11]:
# código base para FAISS
import faiss
import numpy as np

# Asumiendo `embeddings` es un array NxD
D = embeddings.shape[1]
index = faiss.IndexFlatL2(D)
index.add(embeddings)   # los embeddings ya están normalizados

# Usamos el vector de consulta generado anteriormente
query_embedding = query_vec   # shape (1, D)

k = 10
distances, indices = index.search(query_embedding, k)

# Mostrar resultados
print(f"Resultados para la query: '{query_text}'")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"{i+1}. ID: {idx}, Distancia: {dist:.4f}")
    print(f"   Texto: {chunks_df.iloc[idx]['text'][:150]}...")

Resultados para la query: 'Battery measuring'
1. ID: 10176, Distancia: 0.2593
   Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
2. ID: 1, Distancia: 0.2764
   Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
3. ID: 10177, Distancia: 0.3198
   Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
4. ID: 37406, Distancia: 0.3217
   Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...
5. ID: 71872, Distancia: 0.3228
   Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cel

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?

Usamos COSINE (distancia coseno). Elegimos esta métrica porque nuestros embeddings están normalizados (normalize_embeddings=True), lo que hace que la similitud coseno sea equivalente al producto punto y sea la métrica más adecuada para comparar vectores semánticos, ya que mide la orientación (ángulo) entre ellos, no su magnitud.


- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?

Es mucho más sencillo en Qdrant: la API permite agregar un filtro directamente en la consulta (`filter` parameter) usando condiciones sobre el payload (ej. `doc_id=0`). En FAISS no hay soporte nativo, por lo que tendríamos que obtener los vecinos y luego filtrar manualmente en una etapa posterior, lo cual es menos eficiente y requiere código adicional.


- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?

El tiempo de respuesta aumenta ligeramente con `k`, pero no de forma lineal porque el índice ANN (HNSW) ya encuentra a los vecinos cercanos; el incremento se debe principalmente al costo de ordenar y transferir más resultados. Para valores moderados (hasta 100), la diferencia es pequeña; para `k` muy grandes, el impacto puede ser más notorio.



In [12]:
import sys
!{sys.executable} -m pip uninstall -y qdrant-client
!{sys.executable} -m pip install qdrant-client

   ---------------------------------------- 0.0/398.1 kB ? eta -:--:--
   --- ------------------------------------ 30.7/398.1 kB 1.3 MB/s eta 0:00:01
   --------- ------------------------------ 92.2/398.1 kB 1.0 MB/s eta 0:00:01
   ----------------- ---------------------- 174.1/398.1 kB 1.3 MB/s eta 0:00:01
   ------------------------------ --------- 307.2/398.1 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 398.1/398.1 kB 1.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/61.8 kB ? eta -:--:--
   ---------------------------------------- 61.8/61.8 kB ? eta 0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   -- ------------------------------------- 0.4/6.9 MB 13.5 MB/s eta 0:00:01
   ------- -------------------------------- 1.2/6.9 MB 13.1 MB/s eta 0:00:01
   ----------- ---------------------------- 1.9/6.9 MB 13.5 MB/s eta 0:00:01
   ------------- -------------------------- 2.4/6.9 MB 13.8 MB/s eta 0:00:01
   ----------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\FREDDY\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client import models
import numpy as np

# Crear cliente en memoria para no necesitar un servidor externo
client = QdrantClient(":memory:")

# Nombre de la colección
collection_name = "wikipedia_chunks"

# Crear colección con métrica COSINE
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.shape[1],
        distance=models.Distance.COSINE
    )
)

# Preparar puntos para insertar
points = []
for i, (embedding, text) in enumerate(zip(embeddings, chunks_df["text"])):
    points.append(
        models.PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                "text": text,
                "doc_id": int(chunks_df.iloc[i]["doc_id"]),
                "chunk_id": int(chunks_df.iloc[i]["chunk_id"])
            }
        )
    )

# Insertar en lotes (opcional, pero útil para grandes volúmenes)
batch_size = 100
for i in range(0, len(points), batch_size):
    client.upsert(
        collection_name=collection_name,
        points=points[i:i+batch_size]
    )

print(f"Insertados {len(points)} puntos en Qdrant (memoria).")

# Función de búsqueda usando query_points (nuevo método)
def qdrant_search(query_embedding, k=5):
    # Aplanar el vector a lista de floats
    vector_data = query_embedding.flatten().tolist()
    
    search_result = client.query_points(
        collection_name=collection_name,
        query=vector_data,
        limit=k,
        with_payload=True   # para obtener los metadatos
    )
    
    results = []
    for hit in search_result.points:
        results.append({
            "id": hit.id,
            "score": hit.score,
            "text": hit.payload.get("text", "No text"),
            "metadata": {
                "doc_id": hit.payload.get("doc_id"),
                "chunk_id": hit.payload.get("chunk_id")
            }
        })
    return results

# Ejecutar búsqueda
query_embedding = query_vec   # definido anteriormente
results_qdrant = qdrant_search(query_embedding, k=5)

print("\nResultados Qdrant para la query:", query_text)
for res in results_qdrant:
    print(f"ID: {res['id']}, Score: {res['score']:.4f}")
    print(f"Texto: {res['text'][:150]}...")
    print()

C:\Users\FREDDY\AppData\Local\Temp\ipykernel_38652\946137515.py:39: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20100 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Insertados 79104 puntos en Qdrant (memoria).

Resultados Qdrant para la query: Battery measuring
ID: 10176, Score: 0.8703
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

ID: 1, Score: 0.8618
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

ID: 10177, Score: 0.8401
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

ID: 37406, Score: 0.8391
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

ID: 71872, Score: 0.8386
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation pe

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?

En mi implementación usé ChromaDB con su configuración por defecto (índice HNSW y métrica coseno). Chroma no expone parámetros como `ef` o `nprobe` para ajustar directamente, pero internamente utiliza HNSW con valores predeterminados que ofrecen un buen equilibrio. Para simular un ajuste, podría haber modificado el tamaño del lote de inserción o el número de resultados (`k`), pero en esencia la búsqueda es aproximada y rápida. En un sistema como Milvus real, se ajustarían parámetros como `ef` (para HNSW) o `nprobe` (para IVF) para controlar la compensación precisión‑velocidad.


- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?

Aunque en mis pruebas el overlap entre los top‑5 de `k=5` y `k=20` fue del 100%, esto no descarta que ANN pueda cambiar resultados. La evidencia principal es que Chroma utiliza un índice aproximado (HNSW) en lugar de una búsqueda exacta (fuerza bruta). Si comparara con una búsqueda exacta (por ejemplo, usando FAISS con `IndexFlatL2`), podría encontrar diferencias en el orden o en los IDs devueltos, especialmente en los bordes de la lista de resultados. Además, al aumentar `k` se incluyen vecinos más lejanos, que podrían variar ligeramente según cómo se explore el grafo. En mi experimento, la alta consistencia (100% overlap) sugiere que el índice es estable para este dataset y consulta, pero en general ANN es aproximado y los resultados pueden diferir en otros escenarios.

In [19]:
import chromadb
import time
import numpy as np

print("=== Configurando Chroma para la Parte 4 ===\n")

# Crear cliente persistente (para que los datos no se pierdan al reiniciar)
chroma_client = chromadb.PersistentClient(path="chroma_db_part4")

# Eliminar colección si existe
try:
    chroma_client.delete_collection("wikipedia_chunks_part4")
except:
    pass

# Crear colección con métrica coseno (por defecto usa HNSW)
collection = chroma_client.create_collection(
    name="wikipedia_chunks_part4",
    metadata={"hnsw:space": "cosine"}
)

print("Colección creada.")

# Insertar datos en lotes (igual que antes)
batch_size = 1000
total = len(embeddings)
print(f"Insertando {total} registros...")

for i in range(0, total, batch_size):
    end = min(i + batch_size, total)
    ids = [str(j) for j in range(i, end)]
    embeddings_batch = embeddings[i:end].tolist()
    texts_batch = chunks_df.iloc[i:end]["text"].tolist()
    
    collection.add(
        documents=texts_batch,
        embeddings=embeddings_batch,
        ids=ids
    )
    print(f"Insertados {end} de {total}")

print(f"\nInserción completada. Total: {collection.count()}")

# Función de búsqueda (exacta, pero Chroma usa HNSW por defecto)
def chroma_search_ann(query_embedding, k=5, exact=False):
    """
    Parámetros:
    - exact: si True, fuerza búsqueda exacta (solo si el índice lo permite). 
             Chroma no lo expone directamente, pero podemos simularlo.
    """
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    return results

# Simular dos configuraciones: "precisa" (usando el índice por defecto) y "rápida"
# Como Chroma no permite ajustar parámetros de búsqueda, usamos el mismo método
# pero medimos tiempos y comparamos con una búsqueda sobre una muestra más pequeña.

# Configuración por defecto (HNSW con parámetros estándar)
start = time.time()
results_default = chroma_search_ann(query_vec, k=5)
time_default = time.time() - start

# Simular búsqueda "rápida": reducimos el número de candidatos (limitando la colección)
# Esto no es real, pero sirve para ilustrar el concepto.
# En la práctica, Chroma no permite cambiar ef_search, pero podemos crear una colección
# con menos datos para simular menor precisión.
# Para efectos didácticos, usamos la misma búsqueda y comparamos con otra donde limitamos
# el número de resultados a devolver (no es lo mismo, pero muestra la idea).

# Mostrar resultados
print("\nResultados (configuración por defecto):")
for i, (idx, dist, doc) in enumerate(zip(
    results_default["ids"][0],
    results_default["distances"][0],
    results_default["documents"][0]
)):
    print(f"{i+1}. ID: {idx}, Distancia: {dist:.4f}")
    print(f"   Texto: {doc[:100]}...")

print(f"\nTiempo de búsqueda (k=5): {time_default*1000:.2f} ms")

# Comparación con k=20
start = time.time()
results_k20 = chroma_search_ann(query_vec, k=20)
time_k20 = time.time() - start
print(f"Tiempo de búsqueda (k=20): {time_k20*1000:.2f} ms")

# Overlap entre k=5 y k=20 (los primeros 5 de k=20 vs los de k=5)
ids_k5 = set(results_default["ids"][0])
ids_k20_first5 = set(results_k20["ids"][0][:5])
overlap = ids_k5.intersection(ids_k20_first5)
print(f"\nOverlap entre los 5 resultados de k=5 y los primeros 5 de k=20: {len(overlap)} de 5")

=== Configurando Chroma para la Parte 4 ===

Colección creada.
Insertando 79104 registros...
Insertados 1000 de 79104
Insertados 2000 de 79104
Insertados 3000 de 79104
Insertados 4000 de 79104
Insertados 5000 de 79104
Insertados 6000 de 79104
Insertados 7000 de 79104
Insertados 8000 de 79104
Insertados 9000 de 79104
Insertados 10000 de 79104
Insertados 11000 de 79104
Insertados 12000 de 79104
Insertados 13000 de 79104
Insertados 14000 de 79104
Insertados 15000 de 79104
Insertados 16000 de 79104
Insertados 17000 de 79104
Insertados 18000 de 79104
Insertados 19000 de 79104
Insertados 20000 de 79104
Insertados 21000 de 79104
Insertados 22000 de 79104
Insertados 23000 de 79104
Insertados 24000 de 79104
Insertados 25000 de 79104
Insertados 26000 de 79104
Insertados 27000 de 79104
Insertados 28000 de 79104
Insertados 29000 de 79104
Insertados 30000 de 79104
Insertados 31000 de 79104
Insertados 32000 de 79104
Insertados 33000 de 79104
Insertados 34000 de 79104
Insertados 35000 de 79104
Insert

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?
